# Handling large datasets

A large xsnow dataset should normally be converted to a binary archive before it is analyzed repeatedly. This tutorial starts with that basic workflow, then introduces the storage and memory controls that become useful as datasets grow.

## 1. Store large datasets in a binary format

Snow stratigraphy data commonly reach xsnow in one of two forms:

- **PRO and SMET files** are text-based SNOWPACK outputs. They are easy to inspect and exchange, but xsnow must parse and combine them each time they are read.
- **NetCDF and Zarr archives** store array data and metadata in binary form. NetCDF is usually one file; Zarr is a directory or object store containing many binary chunks. Binary archives can be created by converting PRO/SMET files, or written directly by SNOWPACK through its Zarr plugin.

For repeated work with a large dataset, prefer NetCDF or Zarr. In case you work with text-based SNOWPACK output, parsing should become a one-time ingestion step, while later analyses can open only the required part of the archive very fast.

### Small, medium, and large are relative

Dataset size depends on the available memory, the dimensions and variables in the files, and the intended analysis. File count alone is not a reliable measure. Use these categories as workflow descriptions rather than fixed thresholds:

| Dataset | Practical workflow |
|---|---|
| Small | Read PRO/SMET eagerly, analyze in memory, and optionally save an archive |
| Medium | Read PRO lazily once, then save NetCDF or Zarr for repeated use |
| Large | Convert bounded groups of PRO/SMET files and append them to an archive |

For a small source collection, eager loading is simplest:

```python
dataset = xsnow.read("model_output/", lazy=False)
```

For a medium collection, Dask-backed PRO arrays can avoid immediately computing the combined result. Save the result once and use the binary archive afterwards:

```python
dataset = xsnow.read("model_output/", lazy=True)
dataset.to_netcdf("season.nc")
# Or: dataset.to_zarr("season.zarr")
```

When `lazy` is omitted, xsnow chooses a source-specific default:

| Source | Automatic behavior (`lazy=None`) |
|---|---|
| PRO/SMET collection | Eager for at most 30 discovered files; lazy above 30 |
| NetCDF or Zarr archive | Eager when the selected decoded dataset is estimated below 1 GiB; lazy otherwise |

Archive size is estimated after applying `time`, `location`, `slope`, and `realization` selectors. The estimate describes decoded array data rather than compressed size on disk and excludes Python/index overhead and temporary allocations. Archives containing object or variable-length string data, or whose size cannot be estimated safely, remain lazy. Explicit `lazy=True` and `lazy=False` always override these defaults and return Dask-backed and in-memory data, respectively.

In an operational medium-sized domain, `.append_latest()` can read timestamps newer than an existing dataset. Saving the result to NetCDF still rewrites the archive, so this approach is not a bounded in-place update.

```{admonition} Large raw collections: batch conversion
:class: important

For raw PRO/SMET collections that are too large to assemble comfortably, xsnow will provide a batch-append converter. It will parse one bounded group eagerly, write or append that group to a binary archive, release it, and continue with the next group. The same mechanism can append each new day in an operational workflow.

This converter is planned in [issue 23](https://gitlab.com/avacollabra/postprocessing/xsnow/-/work_items/23) and is not yet available. The current PRO `lazy=True` path still parses every source file while constructing the returned dataset; it is useful at medium scale but is not a substitute for bounded conversion of the largest collections.
```

## 2. Understand physical chunks on disk

A multidimensional array is usually not stored as one indivisible block. NetCDF and Zarr divide it into rectangular blocks called **physical chunks**, **storage chunks**, or **encoding chunks**.

For example, a profile variable might be stored in chunks containing:

```text
200 locations x 24 times x 1 slope x 1 realization x all layers
```

Reading one value from that variable generally requires reading and decompressing the physical chunk containing it. Reading many adjacent values from the same chunk is therefore efficient. In Zarr, chunks are stored as separate objects, often files. NetCDF stores them as internal HDF5 blocks.

### When physical chunks are chosen

Physical chunks are fixed when the archive is written. They may be chosen by:

- the SNOWPACK Zarr plugin when SNOWPACK writes a Zarr store directly;
- the planned batch-append converter when xsnow converts PRO/SMET files; or
- an explicit xarray `encoding` passed to the current NetCDF/Zarr writers.

Changing physical chunks later requires rewriting, or rechunking, the archive.

The SNOWPACK plugin currently uses approximately 200 locations, 24 output times, and all layers after benchmarking the operational Avalanche Canada workflow. This is a useful starting point, not necessarily an optimum for every analysis. A time-series workload and an all-locations daily map may need different layouts.

First-class physical chunk controls for `.to_netcdf()` and `.to_zarr()` are planned.

## 3. Understand logical chunks while reading

Physical chunks describe the archive on disk. **Logical chunks**, usually Dask chunks, describe how an opened array is divided into computational tasks. They are a read-time and analysis-time choice.

A logical chunk may match one physical chunk or combine several physical chunks. Combining whole physical chunks often works well because each storage block is read once. A logical chunk that cuts repeatedly across physical boundaries causes inefficiencies.

- Smaller logical chunks reduce the memory required by one task but create more tasks and scheduler overhead.
- Larger logical chunks reduce task count but require more memory per task.

When an array is Dask-backed, `array.chunks` shows its logical chunks. NetCDF's `Variable.chunking()` or Zarr metadata shows physical chunks.

```{admonition} Current `chunks` support
:class: note

`xsnow.read(..., chunks=...)` configures logical Dask chunks for PRO/SMET collections and is forwarded to xarray for NetCDF and Zarr archives. Set `lazy=True` when the returned dataset must remain Dask-backed:

```python
dataset = xsnow.read(
    "season.zarr",
    lazy=True,
    chunks={"location": 400, "time": 48, "layer": -1},
)
```

For archive reads, `chunks=None` lets xarray/Dask choose automatic chunks, while `chunks={}` requests backend-preferred/native chunks. A mapping requests custom logical chunks. The argument is ignored when `lazy=False`; with `lazy=None`, it is used while opening and estimating the archive, but a small selected dataset may still be loaded eagerly. Logical chunk choices do not rewrite physical chunks in the archive.
```

## 4. Understand deferred lazy loading

With deferred loading, `read()` returns the dataset structure without loading every data value into memory. Later selections and calculations describe work to be done. Data is read when an operation actually needs the values.

Explicit `lazy=True` and `lazy=False` have the same public meaning for every supported source, although parsing details still depend on the format:

| Source | Current behavior |
|---|---|
| PRO | Source files are parsed while the dataset is assembled; `lazy=True` returns Dask-backed profile arrays |
| SMET | Parsing is internally eager, but `lazy=True` converts the merged result to Dask-backed arrays |
| NetCDF | Explicit modes are exact; `lazy=None` uses the selected decoded-size estimate |
| Zarr | Uses the same public lazy, eager, automatic-size, and `chunks` behavior as NetCDF |

Lazy loading is therefore most effective for repeated analysis of an existing binary archive. It does not make the current ingestion of an arbitrarily large raw collection memory-bounded.

### Select, reduce, then compute

A useful lazy pipeline becomes smaller before it is computed:

1. **Select** the required locations, times, slopes, realizations, and variables.
2. **Reduce** dimensions with operations such as `mean`, `max`, or `sum`.
3. **Compute** the result once it is small enough for memory.

The following example restricts the archive while opening it, selects one slope, and computes only the time-mean density. Keep the source open until every derived lazy result has been computed:

```python
dataset = xsnow.read(
    "season.zarr",
    location=["VIR1A", "VIR2A"],
    time=slice("2025-01-01", "2025-01-02"),
    lazy=True,
)
try:
    result = (
        dataset["density"]
        .isel(slope=0)
        .mean(dim="time")
        .compute()
    )
finally:
    dataset.close()
```

### Lazy-loading pitfalls

- Lazy does not mean zero memory: metadata, coordinates, task graphs, caches, and computed chunks still occupy memory.
- `.values`, `.load()`, `.compute()`, dataframe conversion, and most plotting eventually read data. Call them after selecting and reducing.
- Too many small logical chunks can create a very large Dask graph.
- A computation can still exceed memory if its final or intermediate result is too large.
- Close a lazy NetCDF or Zarr dataset only after it and any derived lazy objects have been computed or are no longer needed, so backend resources are released.
- Avoid `trim_padding=True` before subsetting a large lazy dataset, because finding all-NaN padding requires computation.

## 5. Handle memory usage

Memory pressure can arise at several different stages:

| Stage | Main control |
|---|---|
| Parsing a large PRO/SMET collection | Conversion batch size |
| Opening or writing many NetCDF variables | `chunk_cache_size` |
| Running Dask tasks | Logical chunk size and number of concurrent tasks |
| Computing a result | Size of the selected and reduced result |

Physical chunks also affect memory because a storage block is the basic unit that must be read and decoded. No one setting caps all memory used by xsnow, xarray, Dask, HDF5, and the final result.

### Read a small subset of a large archive

Apply `time`, `location`, `slope`, and `realization` selectors during `read()`. For both NetCDF and Zarr, xsnow applies these selectors before the automatic size decision or an eager load. A small selection from a large archive can therefore be returned eagerly by default:

```python
subset = xsnow.read(
    "season.zarr",
    time=slice("2025-01-01", "2025-01-07"),
    location=["VIR1A", "VIR2A"],
)
```

Use `lazy=True` when even a small selection should stay Dask-backed, or `lazy=False` when the selected data is known to fit in memory and should be detached from the archive before `read()` returns.

### Limit the NetCDF chunk cache

NetCDF/HDF5 maintains a chunk cache for each open variable. With many variables, those caches can collectively consume substantial memory. Pass `chunk_cache_size` in bytes to change only the cache capacity while xsnow opens or writes the file:

```python
dataset = xsnow.read(
    "season.nc",
    lazy=True,
    chunk_cache_size=1_048_576,  # 1 MiB per variable
)
```

The process default varies between netCDF4 installations. A smaller cache limits this part of memory use but may reduce I/O performance. It does not alter physical or logical chunks.

In [ ]:
import netCDF4

cache_bytes, cache_elements, preemption = netCDF4.get_chunk_cache()
{
    "process_default_cache_MiB": cache_bytes / 1024**2,
    "cache_elements": cache_elements,
    "preemption": preemption,
}

### Planned pre-flight protection

A future pre-flight guard is intended to inspect the estimated decoded size of a selected dataset before xsnow materializes it eagerly. If the estimate exceeds a configured or environment-derived memory budget, xsnow can prevent the allocation and provide actionable guidance, such as narrowing the selection or using lazy loading.

This protection must happen before allocation: an operating-system out-of-memory kill may terminate the process without allowing Python to raise an exception, log a message, or run cleanup code. The exact configuration and behavior of this guard are still being designed.

## 6. Update an operational archive

### when all files are already in binary format
TODO: show example workflow using existing xsnow methods/functions

### when the archive is binary, recent SNOWPACK output is text-based (PRO/SMET)

For a medium domain, the current update pattern is concise:

```python
cached = xsnow.read("season.nc", lazy=False)
updated = cached.append_latest("new_model_output/")
updated.to_netcdf("season.nc")
```

`.append_latest()` reads only timestamps newer than the cached dataset, but the combined dataset and rewritten archive must still fit the workflow. Here `lazy=False` is deliberate: a lazy NetCDF read retains an open backend handle, and overwriting the same file can fail with a permission error. Do not rely on automatic mode for a same-path rewrite because its decision changes with the selected decoded size.

If the existing archive is too large to read eagerly, keep it lazy, write the complete result to a separate temporary archive, finish all deferred reads, close the original dataset, and only then replace the original path.

For larger domains, use the planned batch-append converter to add each new group of PRO/SMET output, or append a Zarr store written directly by the SNOWPACK plugin. Safe appends require monotonic non-overlapping times, compatible dimensions and variables, and batches that align with physical chunks where practical. Each process must own the region it writes; Zarr does not provide general parallel-write locking.

## Summary

1. Prefer a binary NetCDF or Zarr archive for repeated work with large datasets.
2. Automatic mode uses file count for PRO/SMET and selected decoded size for NetCDF/Zarr; use an explicit mode whenever the workflow requires one.
3. Choose physical chunks when writing; they determine storage I/O and require rewriting to change.
4. Choose logical chunks with `chunks` while reading or analyzing; they determine Dask task size without changing archive storage.
5. Use `chunk_cache_size` specifically for NetCDF/HDF5 cache pressure.
6. Select and reduce before computing, close lazy NetCDF and Zarr datasets afterwards, and do not overwrite an archive while it remains lazily open.

See [Reading and writing](reading_writing.ipynb) for general I/O, [Subsetting and filtering](subsetting_filtering.ipynb) for selectors, and the [I/O API reference](../api/_generated/xsnow.io) for complete signatures.